# The gravitational-wave likelihood

Last week was about deciding whether there is a signal in the data. Once we have a detection, we want to know the parameters of the source, e.g. the masses and the distance. Today we derive the likelihood used for this. 

References: Taylor, *The Nanohertz Gravitational Wave Astronomer* ([arXiv:2105.13270](https://arxiv.org/abs/2105.13270)), Sec. 5.3; Maggiore, *Gravitational Waves, Vol. 1*, Sec. 7.4.2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")   # a harmless warning from LAL
import lal
#lal.swig_redirect_standard_output_error(False)   # makes waveform generation ~2x faster inside Jupyter
%config InlineBackend.figure_format = 'retina'

from gwpy.timeseries import TimeSeries
from scipy.signal import welch
from scipy.signal.windows import tukey
from pycbc.waveform import get_fd_waveform
from pycbc.detector import Detector
from pycbc.conversions import mass1_from_mchirp_q, mass2_from_mchirp_q

## Bayes' theorem

$$\underbrace{p(\theta \,|\, d)}_{\text{posterior}} = \frac{\overbrace{p(d \,|\, \theta)}^{\text{likelihood}}\; \overbrace{p(\theta)}^{\text{prior}}}{\underbrace{p(d)}_{\text{evidence}}}$$

The prior $p(\theta)$ is what we know before looking at the data. The likelihood $p(d\,|\,\theta)$ is the probability of the observed data if the parameters were $\theta$, and it is where the detector noise comes in. The evidence $p(d)$ doesn't depend on $\theta$, so for parameter estimation we only need $p(\theta\,|\,d) \propto p(d\,|\,\theta)\, p(\theta)$.

For a binary black hole $\theta$ has 15 parameters (masses, spins, distance, sky position, orientation, time and phase of merger). What we get out is a distribution over all of them, which we usually summarise with a median and a credible interval. To compute it we need the likelihood for GW data.

## A toy likelihood

We know the shape $s_i$ of a signal but not its amplitude $A$, and measure $d_i = A\, s_i + n_i$ with Gaussian noise of standard deviation $\sigma$. If $A$ is right, the residual $d_i - A s_i$ is pure noise. The samples are independent, so their probabilities multiply:
$$p(d\,|\,A) = \prod_{i} \frac{1}{\sqrt{2\pi\sigma^2}}\, e^{-(d_i - A s_i)^2/2\sigma^2}
\quad\Longrightarrow\quad
\ln p(d\,|\,A) = -\frac{1}{2}\sum_{i} \frac{(d_i - A s_i)^2}{\sigma^2} + \text{const}.$$

Notation: we write the likelihood both as $p(d\,|\,\theta)$ and as $L(\theta)$ (or $\mathcal{L}$). They are the same thing.

In [ ]:
SEED = 1
N_SAMPLES = 500
A_TRUE = 0.5
SIGMA = 1.0

rng = np.random.default_rng(SEED)
t = np.linspace(0, 1, N_SAMPLES)
s = np.sin(2 * np.pi * 5 * t)        # known shape of the signal
A_true, sigma = A_TRUE, SIGMA
d = A_true * s + sigma * rng.normal(size=t.size)

A = np.linspace(A_TRUE - 0.5, A_TRUE + 0.5, 500)
logL = np.array([-0.5 * np.sum((d - a * s) ** 2) / sigma**2 for a in A])
posterior = np.exp(logL - logL.max())    # flat prior 
posterior /= np.sum(posterior) * (A[1] - A[0])

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].plot(t, d, color="C0", label="Data", lw=1.5)
axes[0].plot(t, A_true * s, color="crimson", lw=2.5, label="Signal")
axes[0].set_xlabel(r"$t$ $[s]$")
axes[0].grid(alpha=0.5)
axes[0].set_xlim(0, 1)
axes[0].legend(frameon=False, loc="upper left")
axes[1].plot(A, posterior, color="teal", lw=2.5)
axes[1].axvline(A_true, color="orange", ls="--", lw=2.5)
axes[1].set_xlabel(r"$A$")
axes[1].set_ylabel(r"$p(A\,|\,d)$")
axes[1].set_xlim(A_TRUE - 0.5, A_TRUE + 0.5)
axes[1].set_ylim(0)
axes[1].grid(alpha=0.5)
plt.show()

You can't see the signal by eye, but the posterior on $A$ is still fairly narrow, since all 500 samples contribute. The noise moves the peak a bit away from $A_\mathrm{true}$.

We always work with $\ln L$ and take `exp(logL - logL.max())` at the end. For GW data $\ln L \sim -10^4$, so `np.exp(logL)` would just give zero.

Dropping the constant and subtracting $\ln L_\mathrm{max}$ both shift $\ln L$ by an amount $c$ that doesn't depend on $A$:
$$\ln \tilde L(A) = \ln L(A) + c \quad\Longrightarrow\quad \tilde L(A) = e^{c}\, L(A).$$
The factor $e^{c}$ cancels when we normalise the posterior (flat prior):
$$p(A\,|\,d) = \frac{\tilde L(A)}{\int \tilde L(A')\, dA'} = \frac{e^{c}\, L(A)}{e^{c}\int L(A')\, dA'} = \frac{L(A)}{\int L(A')\, dA'}.$$

## The GW likelihood

GW data have the same form, $d(f) = n(f) + h(f;\theta)$, so the likelihood is the probability that the noise equals the residual:
$$p(d\,|\,\theta) = p_\mathrm{noise}\big(n = d - h(\theta)\big).$$

In week 6 we modelled the noise as Gaussian and independent in each frequency bin:
$$n(f_k) = \sigma_k\,(x + i\, y), \qquad \sigma_k^2 = \frac{T\, S_n(f_k)}{4}, \qquad x,\, y \sim \mathcal{N}(0, 1).$$

### Deriving the likelihood

For one bin, the real and imaginary parts are independent Gaussians:
$$p\big(n(f_k)\big) \propto \exp\!\Big[-\frac{|n(f_k)|^2}{2\sigma_k^2}\Big] = \exp\!\Big[-\frac{2\,|n(f_k)|^2}{T\, S_n(f_k)}\Big].$$
Multiplying over all bins, with $\Delta f = 1/T$:
$$p(n) \propto \exp\!\Big[-\frac{1}{2}\; 4\sum_k \frac{|n(f_k)|^2}{S_n(f_k)}\,\Delta f\Big] = \exp\!\Big[-\frac{1}{2}\,(n\,|\,n)\Big],$$
with the inner product from week 3. Setting $n = d - h(\theta)$ (Maggiore eq. 7.63):
$$\boxed{\;\ln p(d\,|\,\theta) = -\frac{1}{2}\,\big(d - h(\theta)\,\big|\,d - h(\theta)\big) + \text{const}\;}$$

## Looking at the likelihood

We inject a signal with chirp mass $\mathcal{M}_c = 26\,M_\odot$, mass ratio $q = m_1/m_2 = 1$ and distance $500$ Mpc. Only $\mathcal{M}_c$, $q$ and $d_L$ are free; everything else is fixed to the true value.

$\ln L$ against chirp mass over a wide range ($q$ and $d_L$ fixed):

<img src="likelihood.png" width="1200">

The peak at the true value is very narrow, and there are secondary peaks on both sides. Changing $\mathcal{M}_c$ changes how fast the frequency sweeps, so the template quickly goes out of phase with the data. At the secondary peaks it is back in phase over part of the signal. Note the scale: $\Delta \ln L = -50$ is already a factor $10^{22}$ in likelihood. That is why the grid in the exercise only covers a narrow range in $\mathcal{M}_c$.

## Priors

In the exercise the priors are flat, so the posterior is just the normalised likelihood. Real analyses use physical priors, for example $p(d_L) \propto d_L^2$ (sources uniform in volume). For a signal this loud the prior hardly affects $\mathcal{M}_c$, but it does affect less well measured parameters like $q$ and $d_L$. Also, the posterior is zero wherever the prior is zero.

## Exercise: posterior on a grid

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")   # a harmless warning from LAL
import matplotlib.pyplot as plt
import numpy as np
from pycbc.waveform import get_fd_waveform
from pycbc.detector import Detector
from pycbc.conversions import mass1_from_mchirp_q, mass2_from_mchirp_q
from itertools import product

Consider a 64 s data stream from the Hanford detector that contains a GW signal from a binary black hole with detector frame chirp mass $M_c = 26 \: M_{\odot}$, mass ratio $q = 1$ at a distance $= 500$ Mpc.

In [ ]:
# --- load the H1 data (noise + signal, in frequency domain) and the psd of the noise ---
data = np.load("data/grid_posterior_data.npz")
freqs = data["freqs"]
data_fd = data["data_fd"]
psd = data["psd"]
df = freqs[1] - freqs[0]

In [ ]:
## -- parameters of the source
params_inj = {"Mc": 26, "q": 1, "dist": 500}   # detector-frame chirp mass (Msun), mass ratio m1/m2, distance (Mpc)
ra = 0.83
dec = 0.77
gps_time = 1186741845
d = Detector("H1")
Fp, Fc = d.antenna_pattern(ra, dec, 0, gps_time)

fs = 4096
f_low = 20      
f_high = 1000
f_final = fs/2
band = (freqs >= f_low) & (freqs <= f_high)

## 2. The likelihood on a grid

Assuming the noise is gaussian, we can compute the likelihood $\mathcal{L}(d|h(\theta))$ for different combinations of parameters $\theta$ in order to find the waveform that gives us the maximum likelihood. Specifically, here we want to 
evaluate the likelihood on a 2D grid of $\mathcal{M}_c$ and $q$. 
If we assume a flat prior the posterior is just goes as  $\propto e^{\ln L}$. 


In [ ]:
def inner_prod(a, b, psd):
    return 4 * np.real(np.sum(a[band] * np.conj(b[band]) / psd[band])) * df

def LogLikelihood(params, data_fd, psd):
    """
    params: dict with the parameters that vary ("Mc", "q"); the others are fixed to params_inj
    data_fd: frequency domain data in which you want to find the gw signal
    psd: psd of the noise
    """
    Mc = params["Mc"]
    q = params["q"]
    dist = params_inj["dist"]     # the distance is fixed
    
    # TODO: Implement the gaussian likelihood calculation. 
    # If you need the individual masses, you can get them from the chirp mass and mass ratio with the function imported above: mass1_from_mchirp_q(), mass2_from_mchirp_q 
    
    return ...

In [ ]:
Mc_grid = np.linspace(25.6, 26.06, 185)
q_grid = np.linspace(1, 1.6, 31)
params_Mc_q = [{"Mc": Mc, "q": q} for Mc, q in product(Mc_grid, q_grid)]
logL_Mc_q = np.array([LogLikelihood(p, data_fd, psd) for p in params_Mc_q]).reshape(len(Mc_grid), len(q_grid))
#We subtract the maximum of ln L before exponentiating to avoid the values to become too small.
post_Mc_q = np.exp(logL_Mc_q - logL_Mc_q.max())

## 3. Corner plot

Off-diagonal: the 2D posterior with the 68% (dark) and 95% (light) credible regions. Diagonal: the 1D posteriors, obtained by summing the 2D posterior over the other parameter, with the 68% credible interval shaded.
The orange lines show the injected values.


In [ ]:
def credible_levels(post, probs):
    """posterior values above which the given fractions of the probability lie"""
    p = np.sort(post.ravel())[::-1]
    cdf = np.cumsum(p) / np.sum(p)
    return [p[np.searchsorted(cdf, prob)] for prob in probs]

grids = [Mc_grid, q_grid]
labels = [r"$\mathcal{M}_c$ ($M_\odot$)", "q"]
true = [params_inj["Mc"], params_inj["q"]]
post_2d = {(1, 0): post_Mc_q.T}   # (row, column): rows = y, columns = x
post_1d = [post_Mc_q.sum(axis=1), post_Mc_q.sum(axis=0)]

with plt.style.context("default"):    # white background, no grid
    fig, axes = plt.subplots(2, 2, figsize=(7, 7))
    for i in range(2):
        for j in range(2):
            ax = axes[i, j]
            if j > i:
                ax.axis("off")
                continue
            if i == j:
                p = post_1d[i] / post_1d[i].max()
                level_68 = credible_levels(p, [0.68])[0]
                ax.plot(grids[i], p, color="tab:blue")
                ax.fill_between(grids[i], p, where=p >= level_68, color="tab:blue", alpha=0.3)
                ax.axvline(true[i], color="tab:orange")
                ax.set_ylim(0, 1.1)
                ax.set_yticks([])
            else:
                p = post_2d[(i, j)]
                level_95, level_68 = credible_levels(p, [0.95, 0.68])
                ax.contourf(grids[j], grids[i], p, levels=[level_95, level_68, p.max()], colors=["#a6c8e8", "#4a86c5"])
                ax.contour(grids[j], grids[i], p, levels=[level_95, level_68], colors="#1f4e8c", linewidths=1)
                ax.axvline(true[j], color="tab:orange")
                ax.axhline(true[i], color="tab:orange")
                ax.set_ylim(grids[i][0], grids[i][-1])
            ax.set_xlim(grids[j][0], grids[j][-1])
            if i == 1:
                ax.set_xlabel(labels[j])
            else:
                ax.set_xticklabels([])
            if j == 0 and i > 0:
                ax.set_ylabel(labels[i])
            elif i != j:
                ax.set_yticklabels([])
    plt.tight_layout()
    plt.show()


## 4. Marginalising over the mass ratio

The mass ratio $q$ is correlated with the chirpmass $\mathcal{M}_c$, so marginalising over it broadens the chirp mass posterior. This is different than just looking at the posterior on $\mathcal{M}_c$ while keeping everything else fixed!

$$ p(\mathcal{M}_c|d) = \int p(\mathcal{M}_c,q|d) \:dq $$

In [ ]:
dMc = Mc_grid[1] - Mc_grid[0]
dq = q_grid[1] - q_grid[0]

post_Mc_fixed = post_Mc_q[:, 0]                 # q fixed to q = 1
post_Mc_marg_q = ...#TODO: Implement the marginalization over q 

plt.plot(Mc_grid, post_Mc_fixed / (np.sum(post_Mc_fixed) * dMc), label="everything fixed except Mc")
plt.plot(Mc_grid, post_Mc_marg_q / (np.sum(post_Mc_marg_q) * dMc), label="marginalised over q")
plt.axvline(params_inj["Mc"], color="crimson", ls=":", label="injected")
plt.xlabel(r"$\mathcal{M}_c$ ($M_\odot$)")
plt.ylabel("posterior")
plt.legend()
plt.show()


## Marginalising and credible regions

To get the posterior of one parameter we integrate over the others (Taylor eq. 5.5):
$$p(x\,|\,d) = \int p(x, y\,|\,d)\, dy \qquad \text{(on a grid: a sum along the } y \text{ axis).}$$
This is not the same as fixing $y$. If $x$ and $y$ are correlated, the marginal comes out broader than the slice.

The 68% credible region is the highest-posterior region that contains 68% of the probability.

In [ ]:
def credible_levels(post, probs):
    # posterior values above which the given fractions of the probability lie
    p = np.sort(post.ravel())[::-1]
    cdf = np.cumsum(p) / np.sum(p)
    return [p[np.searchsorted(cdf, prob)] for prob in probs]

x = y = np.linspace(-4, 4, 300)
dx = x[1] - x[0]
X, Y = np.meshgrid(x, y, indexing="ij")
corr = 0.9
post_2d = np.exp(-0.5 * (X**2 - 2 * corr * X * Y + Y**2) / (1 - corr**2))

slice_y = post_2d[:, len(y) // 2]      # y fixed to its true value, 0
marginal = post_2d.sum(axis=1)         # y summed over

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
level_95, level_68 = credible_levels(post_2d, [0.95, 0.68])
axes[0].contourf(x, y, post_2d.T, levels=[level_95, level_68, post_2d.max()], colors=["grey", "black"])
axes[0].contour(x, y, post_2d.T, levels=[level_95, level_68], colors="grey", linewidths=1)
axes[0].axhline(0, color="crimson", ls="--", lw=2.5)
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].grid(alpha=0.5)
axes[0].set_aspect("equal")

axes[1].plot(x, slice_y / (np.sum(slice_y) * dx), color="orange", lw=2.5, label="Conditional")
axes[1].plot(x, marginal / (np.sum(marginal) * dx), color="teal", lw=2.5, label="Marginal")
axes[1].set_xlabel("x")
axes[1].set_ylabel(r"$p(x\,|\,d)$")
axes[1].set_ylim(0)
axes[1].grid(alpha=0.5)
axes[1].legend(frameon=False)
plt.show()

On the left are the 68% and 95% regions, with the slice at $y = 0$ in orange. On the right, fixing $y$ gives a much narrower (and wrong) posterior for $x$. You saw in the exercise that $\mathcal{M}_c$ and $q$ behave the same way.

With more parameters we use a corner plot, with the 1D marginals on the diagonal and the 2D marginals of each pair below. This is how GW results are usually shown.

## Why not always a grid?

$n$ points per parameter in $D$ dimensions means $n^D$ likelihood evaluations:

In [ ]:
start = time.perf_counter()
for _ in range(50):
    LogLikelihood(params_inj, data_fd, psd)
t_eval = (time.perf_counter() - start) / 50
print(f"One evaluation: {1e3 * t_eval:.1f} ms\n")

for D in [1, 2, 3, 6, 15]:
    print(f"D = {D:2d}:  30^D = {30**D:.0e} evaluations  ->  {30**D * t_eval:.0e} s")

Three parameters take a few minutes, which is fine for the exercise. Fifteen would take much longer than the age of the Universe ($4 \times 10^{17}$ s), and most of those points would land where the posterior is zero anyway. Next week we look at Markov chain Monte Carlo (MCMC), which draws samples from the posterior instead of covering the whole space.